In [1]:
!pip install openai


[notice] A new release of pip is available: 24.1.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
import pandas as pd
import re
import json
import asyncio

env_path = '.env'
load_dotenv(dotenv_path=env_path)

api_key = os.environ.get("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OpenRouter API key not found. Please set OPENROUTER_API_KEY in your .env file.")

In [19]:
df = pd.read_csv("../scraper/investor_articles_full.csv")

df

,body_text,category,link,scraped_at,title
0,"JOHANNESBURG, investor.id —Para pemimpin dan d...",international,https://investor.id/international/418386/ktt-g...,2025-11-21T18:07:41.810501,KTT G20 Tetap Berjalan Tanpa AS dan Trump
1,"JAKARTA,investor.id-Kementerian Komunikasi dan...",business,https://investor.id/business/418353/cloudflare...,2025-11-21T18:07:42.880177,Cloudflare Dituding Lindungi Ribuan Situs Judo...
2,"JAKARTA, investor.id – Konglomerasi bisnis asa...",market,https://investor.id/market/418356/dana-jumbo-p...,2025-11-21T18:07:43.961976,Dana Jumbo Posco untukTender OfferSGRO
3,"JAKARTA, investor.id- Emiten komponen otomotif...",corporate-action,https://investor.id/corporate-action/418373/em...,2025-11-21T18:07:45.091613,Emiten TP Rachmat (DRMA) Mau Akuisisi 82% Saha...
4,"JAKARTA, investor.id- PT Teknologi Karya Digit...",corporate-action,https://investor.id/corporate-action/418360/tr...,2025-11-21T18:07:46.183802,"TRON - NBRI Jalin Kemitraan, Pacu Inovasi Kend..."
5,"JAKARTA, investor.id- Komisaris Independen PT ...",corporate-action,https://investor.id/corporate-action/418359/ko...,2025-11-21T18:07:47.300577,Komisaris Independen Telkom Indonesia (TLKM) M...
6,"JAKARTA, investor.id- Indeks harga saham gabun...",stock,https://investor.id/stock/418382/prediksi-ihsg...,2025-11-21T18:07:48.412238,"Prediksi IHSG dan Rekomendasi Saham Senin, 24 ..."
7,"JAKARTA, investor.id- Indeks harga saham gabun...",stock,https://investor.id/stock/418377/ihsg-turun-ti...,2025-11-21T18:07:49.527607,"IHSG Turun Tipis, 5 Saham Justru Kasih Cuan Be..."
8,"JAKARTA, investor.id- Wakil Menteri Keuangan (...",finance,https://investor.id/finance/418387/penyerapan-...,2025-11-21T18:07:50.607732,"Penyerapan Subsidi Bunga KUR Capai Rp 20,3 Tri..."
9,"JAKARTA, investor.id- Kredit bermasalah yang d...",finance,https://investor.id/finance/418383/npl-kur-228...,2025-11-21T18:07:51.686879,"NPL KUR 2,28% per Oktober 2025"


In [20]:
df["body_text"] = df["body_text"].apply(
    lambda x: re.sub(r"^[^—–-]*[—–-]\s*", "", x)
)

# Jangan run code ini sebelum csv dibaca ulang

In [21]:
df

,body_text,category,link,scraped_at,title
0,Para pemimpin dan delegasi dari negara-negara ...,international,https://investor.id/international/418386/ktt-g...,2025-11-21T18:07:41.810501,KTT G20 Tetap Berjalan Tanpa AS dan Trump
1,Kementerian Komunikasi dan Digital (Kemkomdigi...,business,https://investor.id/business/418353/cloudflare...,2025-11-21T18:07:42.880177,Cloudflare Dituding Lindungi Ribuan Situs Judo...
2,"Konglomerasi bisnis asal Korea Selatan, Posco ...",market,https://investor.id/market/418356/dana-jumbo-p...,2025-11-21T18:07:43.961976,Dana Jumbo Posco untukTender OfferSGRO
3,Emiten komponen otomotif milik konglomerat TP ...,corporate-action,https://investor.id/corporate-action/418373/em...,2025-11-21T18:07:45.091613,Emiten TP Rachmat (DRMA) Mau Akuisisi 82% Saha...
4,PT Teknologi Karya Digital Nusa Tbk (TRON) ata...,corporate-action,https://investor.id/corporate-action/418360/tr...,2025-11-21T18:07:46.183802,"TRON - NBRI Jalin Kemitraan, Pacu Inovasi Kend..."
5,Komisaris Independen PT Telkom Indonesia (Pers...,corporate-action,https://investor.id/corporate-action/418359/ko...,2025-11-21T18:07:47.300577,Komisaris Independen Telkom Indonesia (TLKM) M...
6,Indeks harga saham gabungan (IHSG) diprediksi ...,stock,https://investor.id/stock/418382/prediksi-ihsg...,2025-11-21T18:07:48.412238,"Prediksi IHSG dan Rekomendasi Saham Senin, 24 ..."
7,"Indeks harga saham gabungan (IHSG) hari ini, J...",stock,https://investor.id/stock/418377/ihsg-turun-ti...,2025-11-21T18:07:49.527607,"IHSG Turun Tipis, 5 Saham Justru Kasih Cuan Be..."
8,Wakil Menteri Keuangan (Wamenkeu) Suahasil Naz...,finance,https://investor.id/finance/418387/penyerapan-...,2025-11-21T18:07:50.607732,"Penyerapan Subsidi Bunga KUR Capai Rp 20,3 Tri..."
9,Kredit bermasalah yang dipotret dari rasionon-...,finance,https://investor.id/finance/418383/npl-kur-228...,2025-11-21T18:07:51.686879,"NPL KUR 2,28% per Oktober 2025"


In [38]:
client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    default_headers={"HTTP-Referer": "http://localhost:5000"}
)

In [28]:
ner_prompts = 'Here is a paragraph. Please extract all the named entity in the paragraph with format of [{{"entity": Entity Name 1, "type": Entity Type 1}}, {{"entity": Entity Name 2, "type": Entity Type 2}}, {{"entity": Entity Name 3, "type": Entity Type 3}}, ....]. \n\nParagraph: {para}\n\n Key Words: '
summary_prompts = "Here is a paragraph. Please give a summary of the paragraph. \n\nParagraph: {para}\n\n Summary: "

In [29]:
user_prompts = df['body_text']
system_prompt = "You are the smartest AI assistant"

In [30]:
processed_ner_prompts = []
processed_summary_prompts = []

for i in user_prompts:
    processed_ner_prompts.append(ner_prompts.format(para=i))
    processed_summary_prompts.append(summary_prompts.format(para=i))

print(processed_ner_prompts)
print(processed_summary_prompts)

['Here is a paragraph. Please extract all the named entity in the paragraph with format of [{"entity": Entity Name 1, "type": Entity Type 1}, {"entity": Entity Name 2, "type": Entity Type 2}, {"entity": Entity Name 3, "type": Entity Type 3}, ....]. \n\nParagraph: Para pemimpin dan delegasi dari negara-negara makmur serta negara berkembang dijadwalkan berkumpul pada akhir pekan ini untuk menghadiri penyelenggaraan konferensi tingkat tinggi (KTT) G20 di Johannesburg, Afrika Selatan (Afsel). Namun, pertemuan tahun ini diwarnai aksi boikot dari Presiden Amerika Serikat (AS) Donald Trump dan pemerintahannya, dengan tidak mengirimkan perwakilan sama sekali.\n\nKTT G20 pertama yang digelar di benua Afrika itu akan dihadiri oleh perwakilan dari 42 negara. Absennya AS, salah satu anggota pendiri sekaligus pihak yang seharusnya mengambil alih presidensi bergilir G20 di Johannesburg, menjadi sorotan utama dan mendominasi percakapan menjelang pertemuan.\n\n Key Words: ', 'Here is a paragraph. Plea

In [32]:
ner_res = []
summary_res = []

In [35]:
async def run_prompt(processed_prompt):
    final_prompt = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": processed_prompt}
    ]

    response = await client.chat.completions.create(
        model="qwen/qwen3-30b-a3b-instruct-2507",
        messages=final_prompt
    )

    return {
        "prompt": processed_prompt,
        "answer": response.choices[0].message.content
    }

In [36]:
async def run_ner_prompts():
    tasks = [run_prompt(p) for p in processed_ner_prompts]
    results = await asyncio.gather(*tasks)
    return results

async def run_summary_prompts():
    tasks = [run_prompt(p) for p in processed_summary_prompts]
    results = await asyncio.gather(*tasks)
    return results

async def main():
    results = await asyncio.gather(
        run_ner_prompts(),
        run_summary_prompts()
    )
    return results

In [39]:
res = await main()

In [40]:
files_length = len(os.listdir())

with open(f"{files_length}.json", "w") as f:
    json.dump(res, f, indent=2)